In [ ]:

from datasets import load_dataset, get_dataset_config_names

all_configs = get_dataset_config_names("MMInstruction/M3IT")
print(len(all_configs), all_configs)

ok_configs = []
multi_image_configs = []


for cfg in all_configs:
    try:
        ds = load_dataset("MMInstruction/M3IT", cfg, split="train", streaming=True, trust_remote_code=True)
        ex = next(iter(ds.take(100)))
        val = ex["image_base64_str"]
        if len(val) == 1:
            ok_configs.append(cfg)
            # print(cfg, ex.keys())
            # print("instruction: ", (ex["instruction"]))
            # print("inputs: ", (ex["inputs"]))
            # print("image_base64_str: ", (ex["image_base64_str"]))
            # print("outputs: ", (ex["outputs"]))
            # print("\n")
        elif len(val) > 1:
            multi_image_configs.append(cfg)
            print(f"[SKIP multi-image] {cfg}: image_base64_str is list (len={len(val)})")
            
        else:
            print(f"[SKIP unknown image type] {cfg}: type={type(val)}")
        
    except Exception as e:
        print(f"[SKIP] {cfg}: {e}")
    
    

In [ ]:
redundant_configs = [
    'iqa',
    'iqa-rephrased',
    'mmchat',
]
for cfg in redundant_configs:
    if cfg in ok_configs:
        ok_configs.remove(cfg)

In [ ]:
print("Single image: ", ok_configs)
print(len(ok_configs))
print("Multi image: ", multi_image_configs)
print(len(multi_image_configs))

task_to_datasets = {
"Image Captioning": ["coco", "textcap", "image-paragraph-captioning",],
"Classification": ["coco-goi", "coco-text", "imagenet", "coco-itm", "snli-ve", "mocheg", "iqa"],
"Visual Question Answering": ["vqa-v2", "shapes", "	docvqa", "ocr-vqa", "st-vqa", "	text-vqa", "gqa", ],
"Knowledgeable Visual QA": ["okvqa", "a-okvqa", "science-qa","viquae",],
"Reasoning": ["clevr", "nlvr", "vcr", "visual-mrc", "winoground",],
"Generation": ["vist", "visual-dialog", "multi30k",],
"Chinese": ["fm-iqa","coco-cn", "flickr8k-cn", "chinese-food", "mmchat",],
"Video": ["ss", "ivqa", "msvd-qa", "activitynet-qa", "	msrvtt", "msrvtt-qa",]
}

## [TODO] 將過濾資料進行整理 (建立多輪對話[多圖片])

## 將過濾資料進行整理 (不使用多輪對話[多圖片])

In [ ]:
from pathlib import Path
from io import BytesIO
from PIL import Image
from tqdm import tqdm
import os
import base64
import json
import random
import time
from datasets import load_dataset

Path("./M3IT").mkdir(parents=True, exist_ok=True)


def save_b64_image(b64_str, save_dir, filename):
    # 基本型別檢查
    if not isinstance(b64_str, str):
        raise ValueError(f"invalid b64_str: {b64_str!r}")
    os.makedirs(save_dir, exist_ok=True)
    img_bytes = base64.b64decode(b64_str)
    img = Image.open(BytesIO(img_bytes)).convert("RGB")
    path = os.path.join(save_dir, filename)
    img.save(path, format="JPEG", quality=90, optimize=True)
    return path


def build_m3it_sample(example, i, img_root, source):
    # image_base64_str 可能是 list，也可能是單一字串，這裡統一處理
    img_field = example["image_base64_str"]
    if isinstance(img_field, list):
        if not img_field or img_field[0] is None:
            return None
        b64_str = img_field[0]
    else:
        if img_field is None:
            return None
        b64_str = img_field

    img_filename = f"{i:08d}.jpg"
    img_path = save_b64_image(b64_str, img_root, img_filename)

    instr = (example["instruction"] or "").strip()
    inp = (example["inputs"] or "").strip()

    if inp:
        base_human = instr + "\n" + inp
    else:
        base_human = instr

    r = random.random()
    if r < 0.5:
        human_value = "<image>\n" + base_human
    else:
        human_value = base_human + "\n<image>"

    data = {
        "id": str(i),
        "image": img_path,
        "source": source,
        "conversations": [
            {"from": "human", "value": human_value},
            {"from": "gpt", "value": (example["outputs"] or "").strip()},
        ],
    }
    return data


def load_dataset_with_retry(builder, name, split, max_retries=5, **kwargs):
    """避免 IncompleteRead / ChunkedEncodingError，簡單重試幾次。"""
    for attempt in range(1, max_retries + 1):
        try:
            return load_dataset(builder, name, split=split, **kwargs)
        except Exception as e:
            print(f"[load_dataset retry {attempt}/{max_retries}] {builder}:{name}:{split} failed: {e}")
            if attempt == max_retries:
                raise
            time.sleep(10)  # 等一下再重試


SAMPLE_RATIO = 20
out_path = "./M3IT/chat.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    global_i = 0
    for cfg in tqdm(ok_configs, desc="datasets loading"):
        split = f"train[:{SAMPLE_RATIO}%]"

        # 加 retry，其他參數維持原樣
        ds = load_dataset_with_retry(
            "MMInstruction/M3IT",
            cfg,
            split=split,
            trust_remote_code=True,
        )

        for example in ds:
            data = build_m3it_sample(example, global_i, "./M3IT/images", cfg)
            if data is None:
                continue  # 沒圖或 image_base64_str 有問題就跳過
            f.write(json.dumps(data, ensure_ascii=False) + "\n")
            global_i += 1
